In [0]:
# Read Silver layer
silver_df = spark.table("workspace.medallion_lakehouse.silver_generation")

print(f"Silver rows: {silver_df.count()}")
silver_df.printSchema()
display(silver_df.limit(5))

In [0]:
from pyspark.sql.functions import (
    sum as spark_sum, avg, max as spark_max, min as spark_min, round as spark_round, stddev, col, when, lit
)
from pyspark.sql.window import Window

In [0]:
# Gold Table 1: Daily generation summary
# 1 row per day with total wind and solar generation

gold_daily = silver_df.groupBy("year", "month","day") \
    .agg(
        spark_round(spark_sum("wind_gen_mw"),2).alias("total_wind_gen_mwh"),
        spark_round(spark_sum("solar_gen_mw"),2).alias("total_solar_gen_mwh"),
        spark_round(avg("wind_gen_mw"),2).alias("avg_hourly_wind_mw"),
        spark_round(avg("solar_gen_mw"),2).alias("avg_hourly_solar_mw"),
        spark_round(spark_sum("ercot_load_mw"),2).alias("total_load_mwh"),
        spark_round(avg("wind_capacity_factor"),2).alias("avg_wind_capacity_factor"),
        spark_round(avg("solar_capacity_factor"),2).alias("avg_solar_capacity_factor"),
        spark_max("wind_gen_mw").alias("peak_wind_mw"),
        spark_max("solar_gen_mw").alias("peak_solar_mw")
    ).orderBy("year","month","day")

print(f"Daily summary rows: {gold_daily.count()}")
display(gold_daily.limit(5))

In [0]:
# Gold Table 2 — Hourly capacity factors
# Shows how efficiently wind and solar capacity is being utilized
# Capacity Factor = Actual Generation / Total Installed Capacity * 100

gold_capacity = silver_df.groupBy(
    "year", "month", "hour_of_day"
).agg(
    spark_round(avg("wind_capacity_factor"), 2)
        .alias("avg_wind_cf_pct"),
    spark_round(avg("solar_capacity_factor"), 2)
        .alias("avg_solar_cf_pct"),
    spark_round(avg("wind_gen_mw"), 2)
        .alias("avg_wind_gen_mw"),
    spark_round(avg("solar_gen_mw"), 2)
        .alias("avg_solar_gen_mw"),
    spark_round(
        avg("wind_gen_mw") /
        avg("total_wind_installed_mw") * 100, 2
    ).alias("recalc_wind_cf_pct"),
    spark_round(
        avg("solar_gen_mw") /
        avg("total_solar_installed_mw") * 100, 2
    ).alias("recalc_solar_cf_pct")
).orderBy("year", "month", "hour_of_day")

print(f"Capacity factor rows: {gold_capacity.count()}")
display(gold_capacity.limit(12))

In [0]:
# Write Gold Table 1 — Daily Summary
gold_daily.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "workspace.medallion_lakehouse.gold_daily_summary"
    )
print("Gold daily summary written successfully")

# Write Gold Table 2 — Capacity Factors
gold_capacity.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "workspace.medallion_lakehouse.gold_capacity_factors"
    )
print("Gold capacity factors written successfully")

In [0]:
# Gold Table 3 — Anomaly flags
# Flags hours where wind or solar generation dropped
# more than 2 standard deviations below the monthly average
# This is a statistical anomaly detection pattern used in
# real operational monitoring systems

# Step 1 — Calculate monthly mean and stddev per fuel type
# Window partitioned by year and month

monthly_window = Window.partitionBy("year", "month")

silver_with_stats = silver_df \
    .withColumn("wind_monthly_avg",
        avg(col("wind_gen_mw")).over(monthly_window)) \
    .withColumn("wind_monthly_std",
        stddev(col("wind_gen_mw")).over(monthly_window)) \
    .withColumn("solar_monthly_avg",
        avg(col("solar_gen_mw")).over(monthly_window)) \
    .withColumn("solar_monthly_std",
        stddev(col("solar_gen_mw")).over(monthly_window))

print("Stats calculated successfully")
silver_with_stats.select(
    "timestamp_utc", "wind_gen_mw",
    "wind_monthly_avg", "wind_monthly_std"
).show(3, truncate=False)

In [0]:
# Cell 2 — Add anomaly flags separately
wind_threshold = (
    col("wind_monthly_avg") - lit(2) * col("wind_monthly_std")
)

solar_threshold = (
    col("solar_monthly_avg") - lit(2) * col("solar_monthly_std")
)

gold_anomalies = silver_with_stats \
    .withColumn("wind_anomaly_flag",
        when(col("wind_gen_mw") < wind_threshold,
            lit(True)
        ).otherwise(lit(False))
    ) \
    .withColumn("solar_anomaly_flag",
        when(
            (col("solar_gen_mw") > lit(0)) &
            (col("solar_gen_mw") < solar_threshold),
            lit(True)
        ).otherwise(lit(False))
    ) \
    .withColumn("wind_deviation_sigmas",
        spark_round(
            (col("wind_gen_mw") - col("wind_monthly_avg")) /
            col("wind_monthly_std"), 2
        )
    ) \
    .withColumn("solar_deviation_sigmas",
        spark_round(
            (col("solar_gen_mw") - col("solar_monthly_avg")) /
            col("solar_monthly_std"), 2
        )
    ) \
    .select(
        "timestamp_utc", "year", "month", "day",
        "hour_of_day", "wind_gen_mw", "solar_gen_mw",
        "wind_monthly_avg", "wind_monthly_std",
        "solar_monthly_avg", "solar_monthly_std",
        "wind_anomaly_flag", "solar_anomaly_flag",
        "wind_deviation_sigmas", "solar_deviation_sigmas"
    ) \
    .orderBy("timestamp_utc")

print("Anomaly flags added successfully")
print(f"Total rows: {gold_anomalies.count()}")

In [0]:
# Cell 3 — Summary stats and write to Gold
total_hours = gold_anomalies.count()

wind_anomalies = gold_anomalies.filter(
    col("wind_anomaly_flag") == True
).count()

solar_anomalies = gold_anomalies.filter(
    col("solar_anomaly_flag") == True
).count()

print(f"Total hours analyzed:  {total_hours}")
print(f"Wind anomaly hours:    {wind_anomalies} "
      f"({round(wind_anomalies/total_hours*100, 2)}%)")
print(f"Solar anomaly hours:   {solar_anomalies} "
      f"({round(solar_anomalies/total_hours*100, 2)}%)")

print("\nSample wind anomalies:")
gold_anomalies.filter(col("wind_anomaly_flag") == True) \
    .select(
        "timestamp_utc", "wind_gen_mw",
        "wind_monthly_avg", "wind_deviation_sigmas"
    ).show(5, truncate=False)

# Write Gold Table 3
gold_anomalies.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "workspace.medallion_lakehouse.gold_anomaly_flags"
    )

print("\nGold anomaly flags written successfully")

In [0]:
# Final verification of all tables in our schema
spark.sql("""
    SHOW TABLES IN workspace.medallion_lakehouse
""").show(truncate=False)

In [0]:
# Row count summary across all layers
layers = {
    "bronze_generation": 
        "workspace.medallion_lakehouse.bronze_generation",
    "silver_generation": 
        "workspace.medallion_lakehouse.silver_generation",
    "gold_daily_summary": 
        "workspace.medallion_lakehouse.gold_daily_summary",
    "gold_capacity_factors": 
        "workspace.medallion_lakehouse.gold_capacity_factors",
    "gold_anomaly_flags": 
        "workspace.medallion_lakehouse.gold_anomaly_flags"
}

print("=" * 55)
print(f"{'Layer':<30} {'Rows':>10} {'Columns':>10}")
print("=" * 55)

for name, table in layers.items():
    df = spark.table(table)
    print(f"{name:<30} {df.count():>10} "
          f"{len(df.columns):>10}")

print("=" * 55)
print("Medallion Lakehouse build complete")


In [0]:
# Interesting query for README — top 10 highest wind anomaly events
spark.sql("""
    SELECT
        timestamp_utc,
        wind_gen_mw,
        ROUND(wind_monthly_avg, 0) as monthly_avg_mw,
        wind_deviation_sigmas,
        ROUND(wind_gen_mw / wind_monthly_avg * 100, 1) 
            as pct_of_monthly_avg
    FROM workspace.medallion_lakehouse.gold_anomaly_flags
    WHERE wind_anomaly_flag = true
    ORDER BY wind_deviation_sigmas ASC
    LIMIT 10
""").show(truncate=False)